In [3]:
!pip -q install -U protobuf sentencepiece tokenizers "huggingface_hub>=0.24.0" "diffusers>=0.30.0" "transformers>=4.40.0" accelerate safetensors pillow tqdm

In [1]:
!unzip per_class_prompts.zip

Archive:  per_class_prompts.zip
  inflating: deer.jsonl              
  inflating: frog.jsonl              
  inflating: truck.jsonl             
  inflating: cat.csv                 
  inflating: horse.csv               
  inflating: airplane.csv            
  inflating: ship.csv                
  inflating: automobile.jsonl        
  inflating: deer.csv                
  inflating: bird.csv                
  inflating: airplane.jsonl          
  inflating: truck.csv               
  inflating: automobile.csv          
  inflating: cat.jsonl               
  inflating: ship.jsonl              
  inflating: horse.jsonl             
  inflating: frog.csv                
  inflating: dog.csv                 
  inflating: dog.jsonl               
  inflating: bird.jsonl              


In [4]:

# Option A: interactive (prompts in notebook)
# Fix dependency mismatch


from huggingface_hub import login
login()  # paste your HF token when prompted

# Option B: non-interactive (set token explicitly)
# import os
# os.environ["HF_TOKEN"] = "hf_..."   # <-- put your token here (or load from env/secret store)
# login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)

# Option C: if you already did `huggingface-cli login` in terminal, you can skip login().


In [5]:
import os, json, time, random, glob, math
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple
from PIL import Image
from tqdm.auto import tqdm

import torch

# Speed knobs (safe defaults for high-VRAM GPU boxes)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [6]:
CIFAR10_CLASSES = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

@dataclass
class GenConfig:
    jsonl_paths: List[str]
    out_root: str = "./cifar10_synth"
    samples_per_class: int = 2000
    seed: int = 123

    # Generate high-res, then downsample to CIFAR-10 32x32
    gen_width: int = 512
    gen_height: int = 512
    cifar_size: int = 32

    # Inference
    steps: int = 30
    guidance: float = 7.5

    # Performance
    batch_size: int = 100           # increase on high VRAM (e.g., 8, 12, 16)
    num_gpus: Optional[int] = None # None = use all visible GPUs

CFG = GenConfig(
    jsonl_paths=["airplane.jsonl"],   # <-- change me
    out_root="./cifar10_synth_sdxl",
    samples_per_class=2000,
    seed=123,
    gen_width=512, gen_height=512,
    cifar_size=32,
    steps=30, guidance=7.5,
    batch_size=100,                 # <-- tune for your VRAM
    num_gpus=None,                # use all GPUs by default
)

os.makedirs(CFG.out_root, exist_ok=True)

In [7]:
def read_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON on {path}:{ln}: {e}")

def load_rows(paths: List[str]) -> List[Dict[str, Any]]:
    rows = []
    for p in paths:
        rows.extend(list(read_jsonl(p)))
    return rows

def normalize_label(s: str) -> str:
    return (s or "").strip().lower()

def group_by_class(rows: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    out = {c: [] for c in CIFAR10_CLASSES}
    other: Dict[str, int] = {}
    for r in rows:
        cls = normalize_label(r.get("class_label"))
        if cls in out:
            out[cls].append(r)
        else:
            other[cls] = other.get(cls, 0) + 1
    if other:
        print("Warning: non-CIFAR class_label(s) found (ignored):")
        for k, v in other.items():
            print(f"  - {k!r}: {v}")
    return out

def stable_sample(items: List[Dict[str, Any]], k: int, rng: random.Random) -> List[Dict[str, Any]]:
    if len(items) <= k:
        return list(items)
    idx = list(range(len(items)))
    rng.shuffle(idx)
    return [items[i] for i in idx[:k]]

rows = load_rows(CFG.jsonl_paths)
by_cls = group_by_class(rows)

rng = random.Random(CFG.seed)
sampled: Dict[str, List[Dict[str, Any]]] = {}
for c in CIFAR10_CLASSES:
    sampled[c] = stable_sample(by_cls[c], CFG.samples_per_class, rng)
    print(f"{c:>10}: sampled {len(sampled[c])} / available {len(by_cls[c])}")

  airplane: sampled 1000 / available 1000
automobile: sampled 0 / available 0
      bird: sampled 0 / available 0
       cat: sampled 0 / available 0
      deer: sampled 0 / available 0
       dog: sampled 0 / available 0
      frog: sampled 0 / available 0
     horse: sampled 0 / available 0
      ship: sampled 0 / available 0
     truck: sampled 0 / available 0


In [8]:
def safe_filename(s: str) -> str:
    s = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in (s or ""))
    s = s.strip("_")
    return (s[:120] or "item")

def resize_to_cifar(img: Image.Image, size: int) -> Image.Image:
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img.resize((size, size), resample=Image.Resampling.LANCZOS)

def build_tasks(sampled_by_class: Dict[str, List[Dict[str, Any]]], backend_name: str, cfg: GenConfig):
    tasks = []
    for cls, items in sampled_by_class.items():
        for r in items:
            prompt = r.get("prompt") or ""
            neg = r.get("negative_prompt") or None
            pid = r.get("id", None)

            base = f"{cfg.seed}|{backend_name}|{cls}|{pid}|{prompt}"
            seed = abs(hash(base)) % (2**31 - 1)

            stem = safe_filename(str(pid)) if pid is not None else safe_filename(prompt[:80])

            # Modified paths here to put class folders directly under cifar_hires and cifar32
            hires_dir = os.path.join(cfg.out_root, "cifar_hires", cls)
            cifar_dir = os.path.join(cfg.out_root, "cifar32", cls)

            hires_path = os.path.join(hires_dir, f"{stem}_{seed}.png")
            cifar_path = os.path.join(cifar_dir, f"{stem}_{seed}.png")

            tasks.append({
                "class_label": cls,
                "prompt_id": pid,
                "prompt": prompt,
                "negative_prompt": neg,
                "seed": int(seed),
                "hires_path": hires_path,
                "cifar_path": cifar_path,
            })
    return tasks

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

In [9]:
# ============================================================
# Load HF pipeline" with this SD3.5 version
# (works with your example: StableDiffusion3Pipeline)
# ============================================================

import torch
from diffusers import StableDiffusion3Pipeline

# ---- Backend config ----
BACKEND_KIND = "sd35"
BACKEND_NAME = "sd35_large"
MODEL_ID = "stabilityai/stable-diffusion-3.5-large"

# SD3.5: bfloat16 is typical on high-VRAM GPUs
DTYPE = torch.bfloat16
device = "cuda:0"

assert torch.cuda.is_available(), "CUDA GPU not available."

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
)

pipe = pipe.to(device)
pipe.set_progress_bar_config(disable=True)

# Optional speed/memory tweaks (safe to try; ignore if unavailable)
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass
try:
    if hasattr(pipe, "vae"):
        pipe.vae.enable_slicing()
        pipe.vae.enable_tiling()
except Exception:
    pass

print("Loaded:", MODEL_ID, "dtype:", DTYPE, "device:", device)


model_index.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded: stabilityai/stable-diffusion-3.5-large dtype: torch.bfloat16 device: cuda:0


In [ ]:
# ============================================================
#  "Generate" with this SD3.5-compatible call
# - SD3.5 uses: prompt, num_inference_steps, guidance_scale
# - width/height supported by most pipelines; keep if it works in your env
# - negative_prompt support varies; we pass it only if present
# ============================================================

import os, json, time
from tqdm.auto import tqdm
import torch

#tasks should already be built by your earlier cells:
tasks = build_tasks(sampled, BACKEND_NAME, CFG)

meta_path = os.path.join(CFG.out_root, f"metadata_{BACKEND_NAME}.jsonl")
os.makedirs(CFG.out_root, exist_ok=True)

def _pipe_call_sd35(prompts, negs, gens):
    """
    SD3.5 pipeline call.
    If negative_prompt causes an error in your version, remove that argument.
    """
    kwargs = dict(
        prompt=prompts,
        num_inference_steps=CFG.steps,    # e.g., 28
        guidance_scale=CFG.guidance,      # e.g., 3.5
        generator=gens,
    )
    # Some versions accept width/height; keep them if supported
    kwargs["width"] = CFG.gen_width
    kwargs["height"] = CFG.gen_height

    # Negative prompt support can vary; try passing it if you have it
    if any(n is not None and n != "" for n in negs):
        kwargs["negative_prompt"] = negs

    return pipe(**kwargs)

print(f"Total tasks: {len(tasks)} | batch_size={CFG.batch_size}")
pbar = tqdm(total=len(tasks))

with open(meta_path, "a", encoding="utf-8") as mf:
    for batch in chunked(tasks, CFG.batch_size):
        # resume support
        batch = [t for t in batch if not (os.path.exists(t["hires_path"]) and os.path.exists(t["cifar_path"]))]

        if not batch:
            pbar.update(CFG.batch_size)  # approx
            continue

        for t in batch:
            os.makedirs(os.path.dirname(t["hires_path"]), exist_ok=True)
            os.makedirs(os.path.dirname(t["cifar_path"]), exist_ok=True)

        prompts = [t["prompt"] for t in batch]
        negs = [t["negative_prompt"] for t in batch]
        gens = [torch.Generator(device=device).manual_seed(int(t["seed"])) for t in batch]

        t0 = time.time()
        try:
            with torch.inference_mode():
                out = _pipe_call_sd35(prompts, negs, gens)

            images = out.images

            for tsk, img in zip(batch, images):
                # Save high-res
                img_rgb = img.convert("RGB") if img.mode != "RGB" else img
                img_rgb.save(tsk["hires_path"], format="PNG", optimize=True)

                # Save CIFAR-10 sized (32x32)
                img32 = resize_to_cifar(img_rgb, CFG.cifar_size)
                img32.save(tsk["cifar_path"], format="PNG", optimize=True)

                rec = {
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "backend": BACKEND_NAME,
                    "kind": BACKEND_KIND,
                    "model_id": MODEL_ID,
                    "class_label": tsk["class_label"],
                    "hires_path": tsk["hires_path"],
                    "cifar32_path": tsk["cifar_path"],
                    "seed": tsk["seed"],
                    "steps": CFG.steps,
                    "guidance": CFG.guidance,
                    "gen_width": CFG.gen_width,
                    "gen_height": CFG.gen_height,
                    "cifar_size": CFG.cifar_size,
                    "prompt_id": tsk["prompt_id"],
                    "prompt": tsk["prompt"],
                    "negative_prompt": tsk["negative_prompt"],
                }
                mf.write(json.dumps(rec, ensure_ascii=False) + "\n")
            mf.flush()

        except Exception as e:
            # If the error is about negative_prompt or width/height,
            # remove those keys inside _pipe_call_sd35 and rerun.
            for tsk in batch:
                mf.write(json.dumps({
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "backend": BACKEND_NAME,
                    "kind": BACKEND_KIND,
                    "model_id": MODEL_ID,
                    "class_label": tsk["class_label"],
                    "prompt_id": tsk["prompt_id"],
                    "prompt": tsk["prompt"],
                    "error": repr(e),
                }, ensure_ascii=False) + "\n")
            mf.flush()

        pbar.update(len(batch))

pbar.close()
print(f"Done. Images in: {os.path.join(CFG.out_root, BACKEND_NAME)}")
print(f"Metadata appended to: {meta_path}")


Total tasks: 200 | batch_size=100


  0%|          | 0/200 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (85 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', '<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|

In [ ]:
!nvidia-smi


Mon Jan 19 20:36:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:00:08.0 Off |                    0 |
| N/A   38C    P0             48W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import torch
from transformers import AutoModelForCausalLM

BACKEND_KIND = "hunyuan_image_3"
BACKEND_NAME = "hunyuanimage3"
MODEL_ID = "./HunyuanImage-3"  # local folder

kwargs = dict(
    attn_implementation="sdpa",      # use "flash_attention_2" if installed
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",              # parallel loading across your GPUs
    moe_impl="eager",               # use "flashinfer" if installed
)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
model.load_tokenizer(MODEL_ID)

# Helpful for reproducibility / speed
model.eval()
torch.set_grad_enabled(False)

print("Loaded:", MODEL_ID)
# If available, inspect placement
if hasattr(model, "hf_device_map"):
    print("hf_device_map (sample):", list(model.hf_device_map.items())[:10])


You are using a model of type hunyuan_image_3_moe to instantiate a model of type Hunyuan. This is not supported for all configurations of models and can yield errors.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/32 [00:00<?, ?it/s]

Loaded: ./HunyuanImage-3
hf_device_map (sample): [('vae', 0), ('vision_model', 0), ('vision_aligner', 0), ('timestep_emb', 0), ('patch_embed', 0), ('time_embed', 0), ('final_layer', 0), ('time_embed_2', 0), ('model.wte', 0), ('model.layers.0', 0)]


In [ ]:
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi